# Phase 7: Customer Analysis (RFM Segmentation)

Identifies that 89.5% of revenue runs through one pooled retail till account, then performs RFM (Recency, Frequency, Monetary) segmentation on the 89 identifiable wholesale/trade customer accounts.

## Setup

In [1]:
import pandas as pd, numpy as np, json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'], dtype={'debtor_code':str})
sales = df[df['value_zar'] > 0].copy()

out = '../outputs'

generic_pattern = 'Cash Account|Transfers Customer|Pensioner Discount'
sales['is_generic_account'] = sales['debtor_name'].str.contains(generic_pattern, case=False, regex=True)

## Headline concentration finding (all accounts)

In [2]:
rev_by_acct = sales.groupby(['debtor_code','debtor_name'])['value_zar'].sum().sort_values(ascending=False)
top1_share = rev_by_acct.iloc[0] / rev_by_acct.sum()
print(f"Top account ({rev_by_acct.index[0][1]}) = {top1_share:.1%} of revenue -- a pooled retail till, not one customer.")
print(f"{sales['is_generic_account'].sum():,} of {len(sales):,} sales lines run through generic/pooled accounts "
      f"({sales.loc[sales.is_generic_account,'value_zar'].sum()/sales['value_zar'].sum():.1%} of revenue).")

Top account (Polokwane Cash Account - Butchery) = 89.5% of revenue -- a pooled retail till, not one customer.
526,257 of 535,920 sales lines run through generic/pooled accounts (94.2% of revenue).


## RFM on identifiable (non-generic) wholesale/trade customers only

In [3]:
named = sales[~sales['is_generic_account']].copy()
snapshot_date = named['date'].max() + pd.Timedelta(days=1)

rfm = named.groupby(['debtor_code','debtor_name']).agg(
    recency=('date', lambda x: (snapshot_date - x.max()).days),
    frequency=('doc_number', 'nunique'),
    monetary=('value_zar', 'sum')
).reset_index()

print(f"\nNamed (non-generic) customers analysed: {len(rfm)}")

# Quartile-based RFM scoring (5 = best)
rfm['R'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['R'] + rfm['F'] + rfm['M']

def segment(row):
    if row['rfm_score'] >= 13:
        return 'Champions'
    elif row['rfm_score'] >= 10:
        return 'Loyal'
    elif row['R'] <= 2 and row['M'] >= 4:
        return 'At Risk (high value)'
    elif row['R'] <= 2:
        return 'Churned/Lapsed'
    else:
        return 'Developing'

rfm['segment'] = rfm.apply(segment, axis=1)
rfm = rfm.sort_values('monetary', ascending=False)
rfm.to_csv(f'{out}/customer_rfm.csv', index=False)

seg_summary = rfm.groupby('segment').agg(customers=('debtor_code','count'), total_revenue=('monetary','sum')).sort_values('total_revenue', ascending=False)
seg_summary['pct_of_named_revenue'] = (seg_summary['total_revenue'] / rfm['monetary'].sum() * 100).round(1)
print("\nCustomer segments (named accounts only):")
print(seg_summary)
seg_summary.to_csv(f'{out}/customer_segments_summary.csv')

fig, ax = plt.subplots(figsize=(7,5))
seg_summary['total_revenue'].sort_values().plot(kind='barh', ax=ax, color='#2563eb')
ax.set_title('Revenue by Customer Segment (named accounts)')
ax.set_xlabel('Revenue (ZAR)')
plt.tight_layout()
plt.savefig(f'{out}/customer_segments.png', dpi=150)
plt.close()

# Top 10 named customers table
top10_named = rfm.head(10)[['debtor_name','recency','frequency','monetary','segment']]
top10_named.to_csv(f'{out}/top10_named_customers.csv', index=False)
print("\nTop 10 named customers:")
print(top10_named.to_string(index=False))

summary = {
    'top_account_share_of_total_revenue': float(top1_share),
    'generic_account_lines': int(sales['is_generic_account'].sum()),
    'generic_account_revenue_share': float(sales.loc[sales.is_generic_account,'value_zar'].sum()/sales['value_zar'].sum()),
    'named_customers_analysed': int(len(rfm)),
    'named_customer_total_revenue': float(rfm['monetary'].sum()),
}
with open(f'{out}/customer_analysis_summary.json','w') as f:
    json.dump(summary, f, indent=2)


Named (non-generic) customers analysed: 89

Customer segments (named accounts only):
                      customers  total_revenue  pct_of_named_revenue
segment                                                             
Champions                    17     3725879.54                  65.0
Loyal                        22     1486293.02                  25.9
Developing                   19      246273.88                   4.3
At Risk (high value)          5      143402.84                   2.5
Churned/Lapsed               26      127118.74                   2.2

Top 10 named customers:
                                debtor_name  recency  frequency  monetary   segment
             Polokwane Meat World (Pty) Ltd        2        134 662656.53 Champions
                    Radima Enterprise(CASH)        5        172 538523.81 Champions
                                Duroc Foods      769        127 460041.17     Loyal
        United Food Company (Pty) Ltd(CASH)        1        221 374041